In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print(f'TensorFlow version : {tf.__version__}')
print('Libraries loaded successfully')

In [ ]:
monthly = pd.read_csv('data/processed/monthly_aggregated.csv', parse_dates=['ds'])
monthly = monthly.sort_values('ds').reset_index(drop=True)

print(f'Loaded {len(monthly)} months of data')
print(f'Date range: {monthly["ds"].min().date()} to {monthly["ds"].max().date()}')
print()
monthly[['ds', 'total_qty', 'total_revenue']].tail(6)

### Configure Parameters

In [ ]:
LOOKBACK    = 6    # How many past months LSTM uses to predict next month
TEST_MONTHS = 3     # Oct, Nov, Dec 2025 = test set (same as Prophet)
EPOCHS      = 200   # Max training epochs (EarlyStopping will stop earlier)
BATCH_SIZE  = 8     # Small batch size for small dataset
PATIENCE    = 20    # EarlyStopping patience

print(f'Lookback window : {LOOKBACK} months')
print(f'Test months     : {TEST_MONTHS} (Oct-Dec 2025)')
print(f'Max epochs      : {EPOCHS}')
print(f'Batch size      : {BATCH_SIZE}')

In [ ]:
def create_sequences(data, lookback):
    """
    Convert a 1D time series into (X, y) pairs for LSTM.
    Each X = lookback months of history, y = next month value.
    """
    X, y = [], []
    for i in range(lookback, len(data)):
        X.append(data[i - lookback:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)


def compute_metrics(actual, predicted, label):
    """
    Compute MAE, RMSE, MAPE and print results.
    """
    mae  = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100

    print(f'=== {label} — Test Set Evaluation (Oct-Dec 2025) ===')
    print(f'MAE  : {mae:,.2f}')
    print(f'RMSE : {rmse:,.2f}')
    print(f'MAPE : {mape:.2f}%')
    print()
    return mae, rmse, mape


def build_lstm_model(lookback):
    """
    Stacked LSTM architecture:
    LSTM(64) -> Dropout -> LSTM(32) -> Dropout -> Dense(1)
    """
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(lookback, 1)),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


print('Helper functions defined')

### Data Preparation

In [ ]:
# Extract quantity series
qty_series = monthly['total_qty'].values.reshape(-1, 1)

# Scale to [0, 1] — LSTM performs better on normalized data
scaler_qty = MinMaxScaler(feature_range=(0, 1))
qty_scaled = scaler_qty.fit_transform(qty_series)

# Create sequences
X_qty, y_qty = create_sequences(qty_scaled, LOOKBACK)

# Train/test split — last 3 months = test (same as Prophet)
X_qty_train, X_qty_test = X_qty[:-TEST_MONTHS], X_qty[-TEST_MONTHS:]
y_qty_train, y_qty_test = y_qty[:-TEST_MONTHS], y_qty[-TEST_MONTHS:]

# Reshape for LSTM input: (samples, timesteps, features)
X_qty_train = X_qty_train.reshape(-1, LOOKBACK, 1)
X_qty_test  = X_qty_test.reshape(-1, LOOKBACK, 1)

print(f'Training samples : {X_qty_train.shape[0]}')
print(f'Test samples     : {X_qty_test.shape[0]}')
print(f'Input shape      : {X_qty_train.shape}')

### Build & Train Model for Quantity

In [ ]:
model_qty = build_lstm_model(LOOKBACK)
model_qty.summary()

# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)
]

# Train
history_qty = model_qty.fit(
    X_qty_train, y_qty_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

print(f'Training complete — stopped at epoch {len(history_qty.history["loss"])}')

### Training Loss Plot

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(history_qty.history['loss'], label='Train Loss', color='steelblue', linewidth=2)
plt.plot(history_qty.history['val_loss'], label='Val Loss', color='darkorange', linewidth=2)
plt.title('LSTM Quantity Model — Training Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.tight_layout()
plt.savefig('reports/08_qty_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/08_qty_training_loss.png')

### Evaluate Model on Test Data for Quantity

In [ ]:
# Predict on test set
qty_pred_scaled = model_qty.predict(X_qty_test)
qty_pred = scaler_qty.inverse_transform(qty_pred_scaled).flatten()
qty_actual = scaler_qty.inverse_transform(y_qty_test.reshape(-1, 1)).flatten()

# Test months
test_dates = monthly['ds'].iloc[-TEST_MONTHS:].values

# Metrics
mae_qty, rmse_qty, mape_qty = compute_metrics(qty_actual, qty_pred, 'QUANTITY')

# Month-by-month
print('Month-by-month comparison:')
for i, d in enumerate(pd.to_datetime(test_dates)):
    err = (qty_pred[i] - qty_actual[i]) / qty_actual[i] * 100
    print(f'  {d.strftime("%b %Y")}: Actual={qty_actual[i]:,.0f} | Predicted={qty_pred[i]:,.0f} | Error={err:+.1f}%')

print()
print(f'Prophet Quantity MAPE : 19.08%')
print(f'LSTM   Quantity MAPE  : {mape_qty:.2f}%')
if mape_qty < 19.08:
    print(f'LSTM beats Prophet on Quantity by {19.08 - mape_qty:.2f} percentage points')
else:
    print(f'Prophet beats LSTM on Quantity by {mape_qty - 19.08:.2f} percentage points')

### Prepare Data for Revenue

In [ ]:
# Extract revenue series
rev_series = monthly['total_revenue'].values.reshape(-1, 1)

# Scale
scaler_rev = MinMaxScaler(feature_range=(0, 1))
rev_scaled = scaler_rev.fit_transform(rev_series)

# Create sequences
X_rev, y_rev = create_sequences(rev_scaled, LOOKBACK)

# Train/test split
X_rev_train, X_rev_test = X_rev[:-TEST_MONTHS], X_rev[-TEST_MONTHS:]
y_rev_train, y_rev_test = y_rev[:-TEST_MONTHS], y_rev[-TEST_MONTHS:]

# Reshape
X_rev_train = X_rev_train.reshape(-1, LOOKBACK, 1)
X_rev_test  = X_rev_test.reshape(-1, LOOKBACK, 1)

print(f'Training samples : {X_rev_train.shape[0]}')
print(f'Test samples     : {X_rev_test.shape[0]}')

### Build & Train Model for Revenue

In [ ]:
model_rev = build_lstm_model(LOOKBACK)

history_rev = model_rev.fit(
    X_rev_train, y_rev_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.15,
    callbacks=callbacks,
    verbose=1
)

print(f'Training complete — stopped at epoch {len(history_rev.history["loss"])}')

### Plot Training Loss

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(history_rev.history['loss'], label='Train Loss', color='darkorange', linewidth=2)
plt.plot(history_rev.history['val_loss'], label='Val Loss', color='red', linewidth=2)
plt.title('LSTM Revenue Model — Training Loss', fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.tight_layout()
plt.savefig('reports/09_rev_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

### Evaluate Model on Test Data for Revenue

In [ ]:
# Predict on test set
rev_pred_scaled = model_rev.predict(X_rev_test)
rev_pred = scaler_rev.inverse_transform(rev_pred_scaled).flatten()
rev_actual = scaler_rev.inverse_transform(y_rev_test.reshape(-1, 1)).flatten()

# Metrics
mae_rev, rmse_rev, mape_rev = compute_metrics(rev_actual, rev_pred, 'REVENUE')

# Month-by-month
print('Month-by-month comparison:')
for i, d in enumerate(pd.to_datetime(test_dates)):
    err = (rev_pred[i] - rev_actual[i]) / rev_actual[i] * 100
    print(f'  {d.strftime("%b %Y")}: Actual=LAK {rev_actual[i]:,.0f} | Predicted=LAK {rev_pred[i]:,.0f} | Error={err:+.1f}%')

print()
print(f'Prophet Revenue MAPE : 11.92%')
print(f'LSTM   Revenue MAPE  : {mape_rev:.2f}%')
if mape_rev < 11.92:
    print(f'LSTM beats Prophet on Revenue by {11.92 - mape_rev:.2f} percentage points')
else:
    print(f'Prophet beats LSTM on Revenue by {mape_rev - 11.92:.2f} percentage points')

### Forecast on Jan 2026

In [ ]:
# === QUANTITY FORECAST ===
last_12_qty = qty_scaled[-LOOKBACK:].reshape(1, LOOKBACK, 1)
jan26_qty_scaled = model_qty.predict(last_12_qty)
jan26_qty = scaler_qty.inverse_transform(jan26_qty_scaled)[0][0]

# === REVENUE FORECAST ===
last_12_rev = rev_scaled[-LOOKBACK:].reshape(1, LOOKBACK, 1)
jan26_rev_scaled = model_rev.predict(last_12_rev)
jan26_rev = scaler_rev.inverse_transform(jan26_rev_scaled)[0][0]

print('=' * 55)
print('   LSTM — JANUARY 2026 FORECAST')
print('=' * 55)
print(f'Quantity             : {jan26_qty:,.0f} units')
print(f'Revenue              : LAK {jan26_rev:,.0f}')
print(f'Revenue in Billions  : LAK {jan26_rev/1e9:.2f} Billion')
print()
print('--- Prophet January 2026 Forecast (for comparison) ---')
print(f'Quantity : 843,744 units')
print(f'Revenue  : LAK 240,256,227,466')

### Save the Forecast Data in CSV

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

# Save LSTM January 2026 forecast
lstm_forecast = pd.DataFrame({
    'model'        : ['LSTM'],
    'forecast_month': ['2026-01-01'],
    'qty_forecast' : [jan26_qty],
    'rev_forecast' : [jan26_rev]
})

lstm_forecast.to_csv('data/processed/lstm_forecast.csv', index=False)
print('LSTM forecast saved → data/processed/lstm_forecast.csv')
print(lstm_forecast)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('LSTM — Actual vs Predicted (Test Set: Oct-Dec 2025)', fontsize=14, fontweight='bold')

test_labels = [pd.Timestamp(d).strftime('%b %Y') for d in test_dates]
x = np.arange(len(test_labels))
width = 0.35

# Quantity
axes[0].bar(x - width/2, qty_actual, width, label='Actual', color='steelblue', alpha=0.85)
axes[0].bar(x + width/2, qty_pred,   width, label='LSTM Predicted', color='darkorange', alpha=0.85)
axes[0].set_title(f'Quantity Shipped — MAPE: {mape_qty:.2f}% (Prophet: 19.08%)', fontweight='bold')
axes[0].set_ylabel('Quantity (Units)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(test_labels)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
axes[0].legend()
for i in range(len(test_labels)):
    err = (qty_pred[i] - qty_actual[i]) / qty_actual[i] * 100
    axes[0].text(i + width/2, qty_pred[i]*1.01, f'{err:+.1f}%', ha='center', fontsize=9, color='red')

# Revenue
axes[1].bar(x - width/2, rev_actual, width, label='Actual', color='steelblue', alpha=0.85)
axes[1].bar(x + width/2, rev_pred,   width, label='LSTM Predicted', color='darkorange', alpha=0.85)
axes[1].set_title(f'Revenue (LAK) — MAPE: {mape_rev:.2f}% (Prophet: 11.92%)', fontweight='bold')
axes[1].set_ylabel('Revenue (LAK)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(test_labels)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v/1e9:,.0f}B'))
axes[1].legend()
for i in range(len(test_labels)):
    err = (rev_pred[i] - rev_actual[i]) / rev_actual[i] * 100
    axes[1].text(i + width/2, rev_pred[i]*1.01, f'{err:+.1f}%', ha='center', fontsize=9, color='red')

plt.tight_layout()
plt.savefig('reports/10_lstm_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/10_lstm_actual_vs_predicted.png')

### Save the LSTM model

In [ ]:
import os
os.makedirs('models/saved', exist_ok=True)

model_qty.save('models/saved/lstm_qty_model.keras')
model_rev.save('models/saved/lstm_rev_model.keras')

import pickle
with open('models/saved/scaler_qty.pkl', 'wb') as f:
    pickle.dump(scaler_qty, f)
with open('models/saved/scaler_rev.pkl', 'wb') as f:
    pickle.dump(scaler_rev, f)

print('Models saved:')
print('  models/saved/lstm_qty_model.keras')
print('  models/saved/lstm_rev_model.keras')
print('  models/saved/scaler_qty.pkl')
print('  models/saved/scaler_rev.pkl')

### LSTM MODEL SUMMARY

In [ ]:
print('=' * 55)
print('   LSTM MODEL SUMMARY')
print('=' * 55)
print()
print('MODEL CONFIGURATION:')
print(f'  Architecture    : LSTM(64) -> Dropout(0.2) -> LSTM(32) -> Dropout(0.2) -> Dense(1)')
print(f'  Lookback Window : {LOOKBACK} months')
print(f'  Optimizer       : Adam')
print(f'  Loss Function   : MSE')
print(f'  Callbacks       : EarlyStopping + ReduceLROnPlateau')
print(f'  Scaler          : MinMaxScaler [0, 1]')
print()
print('TEST SET PERFORMANCE (Oct-Dec 2025):')
print(f'  Quantity MAPE : {mape_qty:.2f}%  (Prophet: 19.08%)')
print(f'  Revenue  MAPE : {mape_rev:.2f}%  (Prophet: 11.92%)')
print()
print('JANUARY 2026 FORECAST:')
print(f'  Quantity : {jan26_qty:,.0f} units')
print(f'  Revenue  : LAK {jan26_rev:,.0f}')
print(f'  Revenue  : LAK {jan26_rev/1e9:.2f} Billion')
print()